# Verifiable Quantum Circuit Compilation

## v3 · The 27.30 Public Frontier, Re-verified

Between 3 and 14 November the public line moved from 24.18 to 27.30, on every task at once. invsqrt dropped from 62 to
**46 qubits** with a new arithmetic built from source; DQI got a from-source decoder at **889 qubits** with a higher
extractor cutoff and retained products; chemistry got a new (8, 143, 44) table with **coded angle words** whose loaders
need only 4,038 Toffolis per pass. This version assembles exactly that portfolio, as asteriospanoras published it
(public frontier + one free Toffoli-to-CZ fix in invsqrt), pins every file by SHA-256, re-runs the **unmodified official
verifiers** and writes `/kaggle/working/submission.zip`. No circuit here is ours.

| Task | File | Qubits | Reported cost | Ratio |
|---|---|---:|---:|---:|
| `invsqrt` | `dcde7078` + asteriospanoras' dead-write fix | 46 | 78,719.53 | 26.38 |
| `dqi` | `821bf8d1` | 889 | 3.6021e11 | 67.85 |
| `dfthcsos` | npz `2f0cb894` + loaders `93fffb08…` (λ_eff ≈ 46.897, T 14,738) | 427 | ≈ 2.9513e8 | 11.37 |

Expected geometric mean ≈ **27.30** (v2: 24.18). The cells below recompute every number with this session's verifier run.

**Kaggle setup:** Accelerator *None* (CPU only), Internet off. Inputs: the competition data and the outputs of
[Atakan Aldemir's notebook](https://www.kaggle.com/code/atakanaldemir/qcc-gap-3-residue-repair-and-c43-native-compiler)
(invsqrt, DQI, loaders) and [Heliosli's notebook](https://www.kaggle.com/code/heliosli/the-dataset-is-public-now-enjoy)
(chemistry table); [Areeb Ali's notebook](https://www.kaggle.com/code/areebbb/qai-fewer-toffolis-same-qubits-lb-27-11042)
is a fallback source of the DQI file. Runtime ≈ 20 minutes.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────────────────────
VERIFY = 'full'               # 'full': official verifiers at their default shot counts; 'off': package only
SALT = 'flexonafft-v3'        # local verifier salt; the leaderboard uses its own secret salt
# Pinned bytes (SHA-256 prefixes). Files are looked up by hash in every attached submission.zip, so a source notebook
# that moves on cannot change what is submitted: a missing pin stops the notebook.
PINS = {
    'invsqrt.kmb':             'dcde7078',   # 46-qubit line, before the dead-write fix applied below
    'dqi.kmb':                 '821bf8d1',   # Syed Asad Ali v12 decoder, cutoff 18, 241 retained products
    'dfthcsos/submission.npz': '2f0cb894',   # maoxin123's table, Heliosli's weights
    'dfthcsos/angle_k_0.kmb':  '93fffb08',   # maoxin123's 4,038-Toffoli loaders (the other 14 come from the same zip)
}
REFERENCE_COST = {'invsqrt': 2_076_774.233715282, 'dqi': 24_439_463_583_672.93, 'dfthcsos': 3_355_599_588.3155365}
TASKS = ('invsqrt', 'dqi', 'dfthcsos')

In [ ]:
# ── Session: kickmix and the official verifiers ───────────────────────────────────────────────
import os, sys, glob, json, time, math, shutil, hashlib, zipfile
from pathlib import Path
import pandas as pd

NB_T0 = time.time()
ROOT = Path('/kaggle/working'); ROOT.mkdir(parents=True, exist_ok=True); os.chdir(ROOT)
INPUT = Path('/kaggle/input')
SUB = ROOT / 'submission'

def deep_find(pattern):
    return sorted(Path(p) for p in glob.glob(str(INPUT / '**' / pattern), recursive=True))

KICK = deep_find('kickmix*.so')[0].parent
VERIF = deep_find('verify_submission_invsqrt.py')[0].parent.parent
sys.path[:0] = [str(KICK), str(VERIF)]
import kickmix as km
from invsqrt import verify_submission_invsqrt as v_invsqrt
from dqi import verify_submission_dqi as v_dqi
from dfthcsos import verify as v_dfthcsos
sha = lambda b: hashlib.sha256(b).hexdigest()
print('kickmix', KICK, '| verifiers', VERIF)

## Assemble the portfolio

Every `submission.zip` attached as an input is opened and each member is indexed by name and SHA-256. The pinned files
are taken from whichever source has those exact bytes; the fourteen loaders `angle_k_1…14` come from the same zip as
the pinned `angle_k_0`, so the loader family is never mixed.

In [ ]:
POOL = {}                                         # name -> {sha256: (zip path, bytes)}
for z in deep_find('submission.zip'):
    with zipfile.ZipFile(z) as Z:
        for n in Z.namelist():
            if not n.endswith('/'):
                b = Z.read(n)
                POOL.setdefault(n, {}).setdefault(sha(b), (z, b))
print('attached zips:', *[str(z.relative_to(INPUT)) for z in deep_find('submission.zip')], sep='\n  ')

def pinned(name):
    hits = [(h, v) for h, v in POOL.get(name, {}).items() if h.startswith(PINS[name])]
    assert hits, f'{name} {PINS[name]} not found; available: {[h[:8] for h in POOL.get(name, {})]}'
    return hits[0]

if SUB.exists():
    shutil.rmtree(SUB)
(SUB / 'dfthcsos').mkdir(parents=True)
SOURCE = {}
for name in PINS:
    h, (z, b) = pinned(name)
    (SUB / name).write_bytes(b); SOURCE[name] = str(z.relative_to(INPUT))
loader_zip = Path(INPUT / SOURCE['dfthcsos/angle_k_0.kmb'])
with zipfile.ZipFile(loader_zip) as Z:
    for k in range(1, 15):
        n = f'dfthcsos/angle_k_{k}.kmb'
        (SUB / n).write_bytes(Z.read(n)); SOURCE[n] = str(loader_zip.relative_to(INPUT))
for n in sorted(SOURCE):
    print(f'{n:28s} {sha((SUB / n).read_bytes())[:12]}  from {SOURCE[n]}')

## The dead-write fix (asteriospanoras)

In `dcde7078`, operation 3,292 (`CCX q16 q20 q21`) writes into `q21`, which is touched next only by `HMR q21 b61` at
operation 3,403. Measuring first and applying `CZ q16 q20 if b61` gives the same values, measurement record and phase,
one Toffoli fewer on every shot. The edit is reproduced byte for byte from asteriospanoras' notebook; the asserts
refuse any other circuit.

In [ ]:
c = km.Circuit.from_file(str(SUB / 'invsqrt.kmb'), format='kmb'); c.to_file('/tmp/inv.kmx', format='kmx')
lines = open('/tmp/inv.kmx').read().splitlines()
ol = [i for i, l in enumerate(lines) if l.split() and l.split()[0] not in ('APPEND_TO_REGISTER', 'REGISTER', 'INFO_TAG', 'DEBUG_PRINT')]
i1, i2 = ol[3292], ol[3403]
assert lines[i1] == 'CCX q16 q20 q21' and lines[i2] == 'HMR q21 b61', (lines[i1], lines[i2])
assert not any('b61' in l.split() for l in lines[:i2])
assert not any('q21' in l.split() for l in lines[i1 + 1:i2]), 'q21 is touched between the two operations'
lines = lines[:i1] + ['HMR q21 b61', 'CZ q16 q20 if b61'] + lines[i1 + 1:i2] + lines[i2 + 1:]
open('/tmp/inv_fixed.kmx', 'w').write('\n'.join(lines) + '\n')
km.Circuit.from_file('/tmp/inv_fixed.kmx', format='kmx').to_file(str(SUB / 'invsqrt.kmb'), format='kmb')
print('invsqrt after the fix:', sha((SUB / 'invsqrt.kmb').read_bytes())[:12])

FILES = sorted(p.relative_to(SUB).as_posix() for p in SUB.rglob('*') if p.is_file())
expected = {'invsqrt.kmb', 'dqi.kmb', 'dfthcsos/submission.npz', *(f'dfthcsos/angle_k_{k}.kmb' for k in range(15))}
assert set(FILES) == expected, sorted(set(FILES) ^ expected)
HASHES = {f: sha((SUB / f).read_bytes()) for f in FILES}

## Official verification

The three verifiers run unmodified from the competition's `verifiers/` folder: `invsqrt` checks every
`x ∈ [1, 3,133,452]` 64 times; `dqi` runs 102,400 shots and prices failures through `R(ε₊)`; `dfthcsos` decodes the
angle words from the circuits and checks the factorization against the FeMoCo reference. Any `FAIL` stops the notebook.

In [ ]:
RESULTS = {}
if VERIFY == 'full':
    t0 = time.time()
    RESULTS['dfthcsos'] = v_dfthcsos.verify(str(SUB / 'dfthcsos'), salt=SALT)
    print(f'dfthcsos  {time.time() - t0:7.0f} s', flush=True); t0 = time.time()
    circuit = km.Circuit.from_file(str(SUB / 'dqi.kmb'), format='kmb')
    RESULTS['dqi'] = v_dqi.verify(circuit, v_dqi.ProblemSpecDQI.benchmark_instance(), salt=SALT, max_workers=4)
    del circuit
    print(f'dqi       {time.time() - t0:7.0f} s', flush=True); t0 = time.time()
    circuit = km.Circuit.from_file(str(SUB / 'invsqrt.kmb'), format='kmb')
    RESULTS['invsqrt'] = v_invsqrt.verify(circuit, v_invsqrt.ProblemSpecInvSqrt.for_n_p(10), salt=SALT, max_workers=4)
    del circuit
    print(f'invsqrt   {time.time() - t0:7.0f} s', flush=True)
    for task in TASKS:
        assert RESULTS[task]['status'] == 'PASS', (task, RESULTS[task])
else:
    print('VERIFY = off: packaging without a verifier run')

In [ ]:
def describe(task, r):
    if task == 'invsqrt':
        return f"Q {r['num_qubits']}, Toffolis {r['toffoli_count']:.3f}, worst error/tol {r.get('worst_error_over_tolerance', float('nan')):.5f}"
    if task == 'dqi':
        return f"Q {r['num_qubits']}, T {r['toffoli_count']:.0f}, {r.get('value_errors', 0)} value / {r.get('phase_errors', 0)} phase errors"
    return f"lambda_eff {r['lambda_eff']:.6f}, T {r['toffoli_count']:.0f}, Q {r['num_qubits']}"

if RESULTS:
    rows = [{'task': t, 'cost': float(RESULTS[t]['score']), 'ratio': REFERENCE_COST[t] / float(RESULTS[t]['score']),
             'detail': describe(t, RESULTS[t])} for t in TASKS]
    SUMMARY = pd.DataFrame(rows).set_index('task')
    GM = math.prod(SUMMARY.ratio) ** (1 / 3)
    display(SUMMARY.style.format({'cost': '{:.8g}', 'ratio': '{:.5f}'}))
    print(f'Expected leaderboard score (geometric mean of reference / cost): {GM:.5f}')

## Write the submission

The archive holds the files at its root, as the Evaluation page asks: `invsqrt.kmb`, `dqi.kmb` and `dfthcsos/`.

In [ ]:
ZIP = ROOT / 'submission.zip'
with zipfile.ZipFile(ZIP, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for f in FILES:
        z.write(SUB / f, f)
with zipfile.ZipFile(ZIP) as z:
    assert sorted(z.namelist()) == FILES
shutil.rmtree(SUB)
(ROOT / 'provenance.json').write_text(json.dumps({
    'version': 'v3_public_frontier_27_30', 'salt': SALT, 'verify': VERIFY, 'sources': SOURCE, 'sha256': HASHES,
    'results': {t: {k: v for k, v in r.items() if isinstance(v, (int, float, str))} for t, r in RESULTS.items()},
    'expected_GM': GM if RESULTS else None}, indent=1))
print('SUBMISSION READY:', ZIP, f'{ZIP.stat().st_size / 2**20:.1f} MiB', '| sha256', sha(ZIP.read_bytes())[:16],
      f'| {time.time() - NB_T0:.0f} s total')

# Sources and credits

Every circuit in this submission was written by other participants and is used byte for byte from public notebooks,
except the one-line invsqrt fix, which is asteriospanoras'. The credits follow the authors' own notes (asteriospanoras'
[QCC 27.30 | What Works, What Fails and Why](https://www.kaggle.com/code/asteriospanoras/qcc-27-30-what-works-what-fails-and-why),
section 8, and the notebooks it links); if anyone is missing, please tell us in the comments and we will add them.

- **Assembly followed:** asteriospanoras,
  [QCC 27.30 | Public Frontier + Free CZ Fix in invsq](https://www.kaggle.com/code/asteriospanoras/qcc-27-30-public-frontier-free-cz-fix-in-invsq)
  (the file choice, the pins and the dead-write fix, Apache-2.0).
- **invsqrt** `dcde7078` + fix — the 46-qubit own-circuit line of
  [Syed Asad Ali](https://www.kaggle.com/code/syed3000/vqcc-own-circuit-portfolio-notebook) (from-source builders, Apache-2.0)
  and [Ali Haydar Özdağ](https://www.kaggle.com/code/ahoirg/qcc-submit-verify-and-package)'s 64-row build `4560ac0c`;
  full-domain angle refits by [Ertuğrul Özer](https://www.kaggle.com/code/ozertuu/quantum-ai-27-083-vqcc-dqi-cutoff-17) and
  [Atakan Aldemir](https://www.kaggle.com/code/atakanaldemir/qcc-gap-3-residue-repair-and-c43-native-compiler) (33-angle
  cutting-plane refit `dcde7078`), as shipped by [Alex Chilton](https://www.kaggle.com/code/alexchilton/24-94-lb-verified-three-task-portfolio)
  and Atakan Aldemir; the dead-write fix by asteriospanoras. Oleg Sher's
  [Quantum Circuits from Zero](https://www.kaggle.com/code/sheroleg/quantum-circuits-from-zero-1-sqrt-x-explained) explains the design.
- **DQI** `821bf8d1` — [Areeb Ali](https://www.kaggle.com/code/areebbb/qai-fewer-toffolis-same-qubits-lb-27-11042)'s 241 retained
  products on Syed Asad Ali's v12 from-source decoder at extractor cutoff 18 (Ertuğrul Özer's cutoff idea). Earlier
  decoder lineage as the authors credit it: the team *i want to run gemma 9b*, Heliosli, SomeJoint, jday96314, Kirill
  Tymchishin, Atakan Aldemir, Ali Haydar Özdağ, Isabe11aM00re, MAIDANG, daifanhao, Oleg Sher, maoxin123 (the 392k-Toffoli
  rewrite), busyaprime (Akmal Xodarev), yoyobar, and the Khattar–Gidney paper the decoder follows.
- **dfthcsos** npz `2f0cb894` + loaders `93fffb08…` — [maoxin123](https://www.kaggle.com/code/maoxin123/qcc-v27-4038-toffoli-coded-word-chemistry)'s
  (8, 143, 44) table, Reed–Muller-coded angle words and 4,038-Toffoli loaders (CC0 / Apache-2.0); weights W/H2_I by
  [Heliosli](https://www.kaggle.com/code/heliosli/the-dataset-is-public-now-enjoy) and Atakan Aldemir with
  [prvsiyan](https://www.kaggle.com/code/prvsiyan/verifier-guided-quantum-circuit-optimization)'s refit code; Leo Tang's
  weight continuation; Lera (marculera)'s DF-THC table line ([dfthc-c44-build](https://www.kaggle.com/datasets/marculera/dfthc-c44-build))
  and the authors her NOTICE credits (Ahmed Berat Özer, kozykappa, anazemcev, mrmeows, michaelrustad, phucthaiv02,
  somejoint, gengsr, qwertykeypadapple, lightningv08, syed3000, Heliosli, busyaprime); manwithacat (James Barlow)'s
  dfthc_refit and the [VQCC public circuits](https://www.kaggle.com/datasets/manwithacat/vqcc-apache-public-circuits) dataset.
- **Practice:** Alex Chilton's verified-portfolio approach (pin by SHA-256, re-verify with the stock verifiers);
  manwithacat's daily *RezOps* briefings in the discussions.
- **Tasks, kickmix and verifiers:** Google Quantum AI / Qualtran ([arXiv:2610.23832](https://arxiv.org/abs/2610.23832)).

Each circuit keeps its authors' terms; nothing here is relicensed.